# 05 — Hybrid Recommendation Evaluation

**Purpose:** Establish a reproducible top-N evaluation protocol for a hybrid recommender.

This notebook deliberately separates training interactions from held-out test interactions. It compares:
1. Popularity baseline
2. Content-based ranking
3. Collaborative latent-factor ranking
4. A weighted hybrid ranking

The notebook is a research benchmark. Its implementation should be reconciled with `backend/app/ml/recommender.py` before claiming exact parity with the deployed service.

## 1. Load data and create a per-user holdout

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
# Works when launched from the Spectra root or from the notebooks folder.
if not (PROJECT_ROOT / "datasets" / "ml-1m").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "datasets" / "ml-1m"

movies = pd.read_csv(
    DATA_DIR / "movies.dat", sep="::", engine="python",
    names=["movie_id", "title", "genres"], encoding="latin-1"
)
ratings = pd.read_csv(
    DATA_DIR / "ratings.dat", sep="::", engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"], encoding="latin-1"
)
users = pd.read_csv(
    DATA_DIR / "users.dat", sep="::", engine="python",
    names=["user_id", "gender", "age", "occupation", "zip_code"], encoding="latin-1"
)
print(f"Movies: {len(movies):,} | Ratings: {len(ratings):,} | Users: {len(users):,}")

from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

eligible = ratings.groupby("user_id").size()
eligible = eligible[eligible >= 5].index
data = ratings[ratings.user_id.isin(eligible)].copy()

train_parts, test_parts = [], []
for _, g in data.groupby("user_id", sort=False):
    tr, te = train_test_split(g, test_size=0.2, random_state=42)
    train_parts.append(tr); test_parts.append(te)
train = pd.concat(train_parts, ignore_index=True)
test = pd.concat(test_parts, ignore_index=True)

# Relevance definition for implicit top-N evaluation.
RELEVANCE_THRESHOLD = 4.0
test_relevant = (test[test.rating >= RELEVANCE_THRESHOLD]
                 .groupby("user_id").movie_id.apply(set).to_dict())
train_seen = train.groupby("user_id").movie_id.apply(set).to_dict()

Movies: 3,883 | Ratings: 1,000,209 | Users: 6,040


## 2. Train content and collaborative components using training data only

In [2]:
movie_features = movies.copy()
movie_features["text"] = (
    movie_features.title.fillna("").str.replace(r"\(\d{4}\)", "", regex=True)
    + " " + movie_features.genres.fillna("").str.replace("|", " ", regex=False)
).str.lower()

tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
content_matrix = tfidf.fit_transform(movie_features.text)
movie_ids = movie_features.movie_id.to_numpy()
movie_to_idx = {m:i for i,m in enumerate(movie_ids)}

user_ids = np.sort(train.user_id.unique())
rated_movie_ids = np.sort(train.movie_id.unique())
u_map = {u:i for i,u in enumerate(user_ids)}
m_map = {m:i for i,m in enumerate(rated_movie_ids)}
R = csr_matrix(
    (train.rating.to_numpy(float),
     (train.user_id.map(u_map), train.movie_id.map(m_map))),
    shape=(len(user_ids), len(rated_movie_ids))
)
svd = TruncatedSVD(n_components=min(50, min(R.shape)-1), random_state=42)
U = svd.fit_transform(R)
V = svd.components_.T
item_latent = np.zeros((len(movie_ids), V.shape[1]))
for mid, j in m_map.items():
    item_latent[movie_to_idx[mid]] = V[j]

popularity = train.groupby("movie_id").rating.size().to_dict()

## 3. Define candidate scores and metrics

In [3]:
def user_profile_vector(user_id):
    user_rows = train[train.user_id == user_id]
    idxs, weights = [], []
    for row in user_rows.itertuples():
        if row.movie_id in movie_to_idx:
            idxs.append(movie_to_idx[row.movie_id])
            weights.append(float(row.rating) - 3.0)
    if not idxs:
        return None
    w = np.asarray(weights)
    if np.allclose(np.abs(w).sum(), 0):
        return None
    return np.asarray(content_matrix[idxs].T @ w).ravel() / np.abs(w).sum()

def recommend(user_id, method="hybrid", k=10):
    seen = train_seen.get(user_id, set())
    candidates = np.array([m for m in movie_ids if m not in seen])
    cidx = np.array([movie_to_idx[m] for m in candidates])
    content_scores = np.zeros(len(candidates))
    profile = user_profile_vector(user_id)
    if profile is not None:
        content_scores = np.asarray(content_matrix[cidx] @ profile).ravel()

    collab_scores = np.zeros(len(candidates))
    if user_id in u_map:
        user_vec = U[u_map[user_id]]
        for pos, mid in enumerate(candidates):
            if mid in m_map:
                collab_scores[pos] = user_vec @ V[m_map[mid]]

    pop_scores = np.array([popularity.get(m, 0) for m in candidates], dtype=float)
    def norm(x):
        if x.max() == x.min(): return np.zeros_like(x)
        return (x - x.min()) / (x.max() - x.min())

    if method == "content":
        scores = norm(content_scores)
    elif method == "collaborative":
        scores = norm(collab_scores)
    elif method == "popularity":
        scores = norm(pop_scores)
    else:
        scores = 0.5 * norm(content_scores) + 0.5 * norm(collab_scores)
    return candidates[np.argsort(scores)[::-1][:k]]

def precision_recall_ndcg(recommended, relevant):
    if not relevant: return None
    hits = np.array([1 if m in relevant else 0 for m in recommended])
    precision = hits.sum() / max(len(recommended), 1)
    recall = hits.sum() / len(relevant)
    discounts = 1 / np.log2(np.arange(2, len(hits)+2))
    dcg = np.sum(hits * discounts)
    ideal_hits = np.ones(min(len(relevant), len(hits)))
    idcg = np.sum(ideal_hits / np.log2(np.arange(2, len(ideal_hits)+2)))
    return precision, recall, (dcg / idcg if idcg else 0.0)

## 4. Evaluate on users with relevant held-out items

In [4]:
eval_users = [u for u, rel in test_relevant.items()
              if rel and u in train_seen and u in u_map]
# Sample a subset to keep notebook runtime reasonable; increase for final reporting.
rng = np.random.RandomState(42)
eval_users = rng.choice(eval_users, size=min(1000, len(eval_users)), replace=False)

methods = ["popularity", "content", "collaborative", "hybrid"]
rows = []
for method in methods:
    metrics = []
    for user_id in eval_users:
        recs = recommend(int(user_id), method=method, k=10)
        result = precision_recall_ndcg(recs, test_relevant[int(user_id)])
        if result is not None:
            metrics.append(result)
    arr = np.asarray(metrics)
    rows.append({
        "method": method,
        "Precision@10": arr[:,0].mean(),
        "Recall@10": arr[:,1].mean(),
        "NDCG@10": arr[:,2].mean(),
        "evaluated_users": len(arr)
    })
results = pd.DataFrame(rows).sort_values("NDCG@10", ascending=False)
display(results)

,method,Precision@10,Recall@10,NDCG@10,evaluated_users
2,collaborative,0.2643,0.192237,0.320197,1000
3,hybrid,0.1694,0.128804,0.215855,1000
0,popularity,0.1420,0.087339,0.169493,1000
1,content,0.0306,0.026041,0.035428,1000


## 5. Read the results responsibly

- A hybrid model is not automatically better; use the measured results.
- This protocol uses ratings >= 4 as relevant. That threshold is a modelling choice and should be justified.
- The test set is per-user random holdout, not a chronological future-period test.
- Metrics depend on candidate construction and catalogue coverage.
- For rigorous comparisons, repeat across multiple seeds and report uncertainty.
- The content feature space here is MovieLens title + genres. TMDB enrichment is a later experiment, not silently included here.